# Use Case and Model  Life cycle Governance with SageMaker Model Registry resource sharing

## ML Flow Experimentation with Shared Model Group

This notebook is intended to run in Amazon SageMaker Studio with a recent SageMaker Distribution container and Python 3 kernel. It requires `mlflow>=3`, `sagemaker>=3` (SageMaker Python SDK v3), and `sagemaker-mlflow>=0.5.0`, installed from `requirements.txt` below.

In [ ]:
!pip install -r requirements.txt

### 1. Set-up

In [ ]:
import boto3
import mlflow
import pandas as pd
import os
import json
from time import gmtime, strftime
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
bucket_name = sess.default_bucket()
prefix = "mlflow-credit-risk"

sagemaker_client = boto3.client("sagemaker")

s3_root_folder = f"s3://{bucket_name}/{prefix}"

role = get_execution_role(sess)
print (f"Your Amazon SageMaker Execution role is: {role}")

**Access Model Package Groups in Shared Services account**

To be able to access Model Package Groups in a Shared Services AWS account, you'll need the following permissions assigned to the SageMaker execution role. 
Replace **\<YOUR_AWS_ACCOUNT\>** with your own AWS Account number. Replace **\<SHARED_SERVICES_ACCOUNT\>** with the Account number of the Shared Services account.
```json


    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "ram:GetResourceShareInvitations"
            ],
            "Resource": "arn:aws:ram:us-east-1:<YOUR_AWS_ACCOUNT>:resource-share-invitation/*"
        },
        {
            "Effect": "Allow",
            "Action": [
                "ram:AcceptResourceShareInvitation"
            ],
            "Resource": "arn:aws:ram:us-east-1:<SHARED_SERVICES_ACCOUNT>:resource-share-invitation/*"
        }
    ]
}
```

Before you get started, check if there are any pending invitations from the shared services account 
and accept them. 
This will allow you to discover share model package groups and register your model versions against them.

In [ ]:
ram_client = boto3.client('ram')
response = ram_client.get_resource_share_invitations()
pending_invitations = []
# Review all pending invitations
for i in response['resourceShareInvitations']:
    if i['status'] == "PENDING":
        pending_invitations.append(i)
print(pending_invitations,sep='\n')

In [ ]:
# Accept the resource share invitation from shared services account
if pending_invitations:
    response = ram_client.accept_resource_share_invitation(resourceShareInvitationArn=pending_invitations[0]['resourceShareInvitationArn'])
    print(response)

To set up and manage an MLflow App, as well as work with managed MLflow experiments, you'll need the following permissions assigned to the SageMaker execution role

```json
{
	"Version": "2012-10-17",
	"Statement": [
		{
			"Sid": "VisualEditor0",
			"Effect": "Allow",
			"Action": [
				"sagemaker:CreateMlflowApp",
				"sagemaker:DescribeMlflowApp",
				"sagemaker:ListMlflowApps",
				"sagemaker:UpdateMlflowApp",
				"sagemaker:DeleteMlflowApp",
				"sagemaker:CreatePresignedMlflowAppUrl"
			],
			"Resource": "*"
		},
		{
			"Sid": "VisualEditor1",
			"Effect": "Allow",
			"Action": [
				"sagemaker-mlflow:*"
			],
			"Resource": "*"
		}
	]
}
```

In [ ]:
NOTEBOOK_METADATA_FILE = "/opt/ml/metadata/resource-metadata.json"
domain_id = 'default'
if os.path.exists(NOTEBOOK_METADATA_FILE):
    with open(NOTEBOOK_METADATA_FILE, "rb") as f:
        metadata = json.loads(f.read())
        domain_id = metadata.get('DomainId')
        space_name = metadata.get('SpaceName')
        print(f"SageMaker domain id: {domain_id}")

In [ ]:
def get_running_mlflow_app(sagemaker_client, status_filter=['Created', 'Creating']):
    for status in status_filter:
        apps = sagemaker_client.list_mlflow_apps(Status=status, SortBy='CreationTime', SortOrder='Descending')['Summaries']
        if apps:
            for app in apps:
                print(f"Found an MLflow app {app['Arn']} in the status '{status}'.")
                return app['Arn'], app['Name']
    print("No MLflow apps found.")
    return None, None

def create_mlflow_app(sagemaker_client, bucket_name, sm_role, domain_id):
    """
    Creates a new MLflow app and returns its ARN and name.
    """
    timestamp = strftime('%d-%H-%M-%S', gmtime())
    mlflow_name = f"mlflow-{domain_id}-{timestamp}"
    response = sagemaker_client.create_mlflow_app(
        Name=mlflow_name,
        ArtifactStoreUri=f"s3://{bucket_name}/mlflow/{timestamp}",
        RoleArn=sm_role,
        ModelRegistrationMode='AutoModelRegistrationEnabled',
    )

    mlflow_arn = response['Arn']
    print(f"App creation request succeeded. The MLflow app {mlflow_arn} is being created.")
    return mlflow_arn, mlflow_name

# Get a running MLflow app or create a new one if none exists
mlflow_arn, mlflow_name = get_running_mlflow_app(sagemaker_client)
if not mlflow_arn:
    mlflow_arn, mlflow_name = create_mlflow_app(sagemaker_client, bucket_name, role, domain_id)
print(f"Using MLflow app {mlflow_name}")

### 2. Prepare the data

The code was adapted from this repository https://github.com/aws-samples/amazon-sagemaker-credit-risk-prediction-explainability-bias-detection/tree/main

In [ ]:
from sagemaker.core.s3 import S3Downloader
S3Downloader.download(
    "s3://sagemaker-sample-files/datasets/tabular/uci_statlog_german_credit_data/SouthGermanCredit.asc",
    "data",
)

In [ ]:
credit_columns = [
    "status",
    "duration",
    "credit_history",
    "purpose",
    "amount",
    "savings",
    "employment_duration",
    "installment_rate",
    "personal_status_sex",
    "other_debtors",
    "present_residence",
    "property",
    "age",
    "other_installment_plans",
    "housing",
    "number_credits",
    "job",
    "people_liable",
    "telephone",
    "foreign_worker",
    "credit_risk",
]

In [ ]:
training_data = pd.read_csv(
    "data/SouthGermanCredit.asc",
    names=credit_columns,
    header=0,
    sep=r" ",
    engine="python",
    na_values="?",
).dropna()

In [ ]:
test_data = training_data.sample(frac=0.1, random_state=42)
test_data = test_data.drop("credit_risk", axis=1)
test_columns = [
    "status",
    "duration",
    "credit_history",
    "purpose",
    "amount",
    "savings",
    "employment_duration",
    "installment_rate",
    "personal_status_sex",
    "other_debtors",
    "present_residence",
    "property",
    "age",
    "other_installment_plans",
    "housing",
    "number_credits",
    "job",
    "people_liable",
    "telephone",
    "foreign_worker",
]

training_data.to_csv("train.csv", index=False, header=True, columns=credit_columns)
test_data.to_csv("test.csv", index=False, header=True, columns=test_columns)

# save the datasets in S3 for future use
train_s3_url = sess.upload_data(
    path="train.csv",
    bucket=bucket_name,
    key_prefix=f"{prefix}/input"
)
print(f"Upload the dataset to {train_s3_url}")

test_s3_url = sess.upload_data(
    path="test.csv",
    bucket=bucket_name,
    key_prefix=f"{prefix}/input"
)
print(f"Upload the dataset to {test_s3_url}")


### 3. Process the data with Amazon SageMaker

In [ ]:
from time import gmtime, strftime, sleep

experiment_suffix = strftime('%d-%H-%M-%S', gmtime())
registered_model_name = f"credit-risk-model-{experiment_suffix}"
experiment_name = f"credit-risk-model-experiment-{experiment_suffix}"
print(experiment_name)

In [ ]:
mlflow_arn

In [ ]:
import warnings
import pandas as pd
import numpy as np
import tarfile
import sklearn
import joblib
import mlflow
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import make_column_transformer

from sklearn.exceptions import DataConversionWarning
from sagemaker.core.remote_function import remote


@remote(s3_root_uri=f"s3://{bucket_name}/{prefix}", dependencies=f"requirements.txt", instance_type="ml.m5.large")
def preprocess(df, experiment_name, mlflow_arn, bucket_name, prefix, run_id=None):
    """
    Preprocess the input data and split it into training and validation sets.

    Args:
        df (pandas.DataFrame): Input data.
        experiment_name (str): Name of the MLflow experiment.
        run_id (str, optional): MLflow run ID. If not provided, a new run will be created.
        mlflow_arn (str, optional): MLflow tracking URI.
        s3_root_folder (str, optional): S3 root folder for remote execution.

    Returns:
        tuple: A tuple containing the training and validation features and labels.
    """
    try:
        mlflow.set_tracking_uri(mlflow_arn)
        suffix = strftime('%d-%H-%M-%S', gmtime())
        mlflow.set_experiment(experiment_name=experiment_name if experiment_name else f"credit-risk-model-experiment-{suffix}")
        run = mlflow.start_run(run_id=run_id) if run_id else mlflow.start_run(run_name=f"remote-processing-{suffix}", nested=True)

        output_path = "/opt/ml/output/data"
        os.makedirs(output_path, exist_ok=True)

        print("Reading input data")
        model_dataset = mlflow.data.from_pandas(df)
        mlflow.log_input(model_dataset, context="model_dataset")

        print("Performing one-hot encoding")
        categorical_cols = [
            "credit_history",
            "purpose",
            "personal_status_sex",
            "other_debtors",
            "property",
            "other_installment_plans",
            "housing",
            "job",
            "telephone",
            "foreign_worker",
        ]
        transformer = make_column_transformer(
            (OneHotEncoder(sparse_output=False), categorical_cols),
            remainder="passthrough",
        )

        print("Preparing features and labels")
        X = df.drop("credit_risk", axis=1)
        y = df["credit_risk"]

        print("Building scikit-learn transformer")
        featurizer_model = transformer.fit(X)
        features = featurizer_model.transform(X)
        labels = LabelEncoder().fit_transform(y)

        split_ratio = 0.3
        print(f"Splitting data into train and validation sets with ratio {split_ratio}")
        X_train, X_val, y_train, y_val = train_test_split(
            features, labels, test_size=split_ratio, random_state=0
        )

        print(f"Train features shape after preprocessing: {X_train.shape}")
        print(f"Validation features shape after preprocessing: {X_val.shape}")

        mlflow.log_params({"train_shape": X_train.shape, "val_shape": X_val.shape})

        train_features_path = os.path.join(output_path, "train_features.csv")
        print(f"Saving training features to {train_features_path}")
        pd.DataFrame(X_train).to_csv(train_features_path, header=False, index=False)

        train_labels_path = os.path.join(output_path, "train_labels.csv")
        print(f"Saving training labels to {train_labels_path}")
        pd.DataFrame(y_train).to_csv(train_labels_path, header=False, index=False)

        val_features_path = os.path.join(output_path, "val_features.csv")
        print(f"Saving validation features to {val_features_path}")
        pd.DataFrame(X_val).to_csv(val_features_path, header=False, index=False)

        val_labels_path = os.path.join(output_path, "val_labels.csv")
        print(f"Saving validation labels to {val_labels_path}")
        pd.DataFrame(y_val).to_csv(val_labels_path, header=False, index=False)

        model_dir = "/opt/ml/model"
        os.makedirs(model_dir, exist_ok=True)
        model_path = os.path.join(model_dir, "model.joblib")
        model_output_path = os.path.join(model_dir, "model.tar.gz")

        print(f"Saving featurizer model to {model_output_path}")
        joblib.dump(featurizer_model, model_path)
        with tarfile.open(model_output_path, "w:gz") as tar:
            tar.add(model_path, arcname="model.joblib")

        mlflow.sklearn.log_model(
            sk_model=featurizer_model,
            name="processing/model",
            registered_model_name="sk-learn-model",
        )  
        return X_train, X_val, y_train, y_val
        
    except Exception as e:
        print(f"Exception in processing script: {e}")
        raise e
    finally:
        mlflow.end_run()

In [ ]:
df = pd.read_csv("train.csv", names=None, header=0, sep=",")
X_train, X_val, y_train, y_val = preprocess(df, experiment_name, mlflow_arn, bucket_name, prefix)

### 4. Model training with SageMaker training jobs

In [ ]:
import xgboost
import os
import mlflow

@remote(s3_root_uri=f"s3://{bucket_name}/{prefix}", dependencies=f"requirements.txt", instance_type="ml.m5.large")
def train(X, val_X, y, val_y, num_round, params, mlflow_arn, experiment_name,run_id=None):
    """
    Train an XGBoost model and log it to MLflow.

    Returns:
        str: The MLflow logged model ID (for example "m-abc123..."). It is used later to
             attach an inference specification and to register the model.
    """
    mlflow.set_tracking_uri(mlflow_arn)
    # Autolog params and metrics only: the model is logged explicitly below so that we
    # get back a logged model ID to attach the inference specification to.
    mlflow.autolog(log_models=False)
    
    suffix = strftime('%d-%H-%M-%S', gmtime())
    mlflow.set_experiment(experiment_name=experiment_name if experiment_name else f"credit-risk-model-experiment-{suffix}")
    run = mlflow.start_run(run_id=run_id) if run_id else mlflow.start_run(run_name=f"remote-training-{suffix}", nested=True)

    try:
        dtrain = xgboost.DMatrix(X, label=y)
        dval = xgboost.DMatrix(val_X, label=val_y)

        watchlist = [(dtrain, "train"), (dval, "validation")]
        mlflow.log_params(params)

        print("Training the model")
        bst = xgboost.train(
            params=params, dtrain=dtrain, evals=watchlist, num_boost_round=num_round
        )

        # Log the Booster as an MLflow model. MLflow writes model.ubj together with the
        # MLmodel metadata into the logged model's artifact location. That location is
        # exactly what the inference specification will point SageMaker at, so there is
        # no tarball to build and no artifact copy to keep in sync.
        model_info = mlflow.xgboost.log_model(bst, name="model")
        print(f"Logged model ID: {model_info.model_id}")
        return model_info.model_id

    except Exception as e:
        print(f"Exception in training script: {e}")
        raise e
    finally:
        mlflow.end_run()


In [ ]:
hyperparameters = {
    "max_depth": "5",
    "eta": "0.1",
    "gamma": "4",
    "min_child_weight": "6",
    "silent": "1",
    "objective": "binary:logistic",
    "num_round": "100",
    "subsample": "0.8",
    "eval_metric": "auc"
}
num_round = 50

mlflow_model_id = train(X_train, X_val, y_train, y_val,num_round, hyperparameters, mlflow_arn, experiment_name)
print(f"MLflow logged model ID: {mlflow_model_id}")

### 5. Register the candidate model to the model registry in the shared services account

Register the trained model into the shared Model Package Group, with an **inference
specification** attached so the resulting Model Package version is deployable as-is.

The specification is logged onto the MLflow model with `sagemaker_mlflow.log_inference_specification()`
(`sagemaker-mlflow >= 0.5.0`). It records the serving container image and an `S3Prefix`
`ModelDataSource` pointing at the MLflow artifact location, so the endpoint serves
**directly from the MLflow artifact store** — no tarball repacking and no second copy of
the model bytes to keep in sync. The spec also shows up as a
`sagemaker_inference_specification.json` artifact next to the model in the MLflow UI.

**A note on the shared group:** when MLflow Model Registry sync is enabled, calling
`mlflow.register_model()` automatically creates a Model Package Group *in this account*,
named after the registered model. It cannot target a Model Package Group that another
account shared with you. So to register into the Shared Services group we call
`create_model_package` explicitly against the shared group ARN, reusing the very same
specification we logged to MLflow.

In [ ]:
mlflow.set_tracking_uri(mlflow_arn)

#### Step 1: Locate the logged model in the MLflow artifact store

In [ ]:
from mlflow import MlflowClient

mlflow_client = MlflowClient()

# The training job returned the logged model ID. If instead you ran several training
# jobs and want the best one, rank the experiment's logged models on a model metric:
#   candidates = mlflow.search_logged_models(
#       experiment_ids=[mlflow.get_experiment_by_name(experiment_name).experiment_id],
#       order_by=[{"field_name": "metrics.validation-auc", "ascending": False}],
#       max_results=1,
#   )
logged_model = mlflow_client.get_logged_model(mlflow_model_id)

print(f"Logged model:      {logged_model.model_id}")
print(f"Artifact location: {logged_model.artifact_location}")

#### Step 2: Resolve the serving container image

In [ ]:
from sagemaker.core import image_uris

instance_type = "ml.m5.xlarge"

# Resolve the managed XGBoost serving image for the current region instead of
# hardcoding an ECR URI or image digest.
xgboost_image = image_uris.retrieve(
    framework="xgboost",
    region=sess.boto_region_name,
    version="1.7-1",
    image_scope="inference",
    instance_type=instance_type,
)
print(f"Inference container image: {xgboost_image}")

#### Step 3: Log a minimal `inference.py` alongside the model

In [ ]:
import os
import tempfile

# The XGBoost serving container needs a model_fn to load the model MLflow wrote
# (model.ubj). We log a small code/inference.py as an artifact of the logged model, so
# the S3Prefix ModelDataSource brings it along and it lands at
# /opt/ml/model/code/inference.py on the endpoint.
INFERENCE_SCRIPT = """\
import os

import xgboost


def model_fn(model_dir):
    booster = xgboost.Booster()
    booster.load_model(os.path.join(model_dir, "model.ubj"))
    return booster
"""

# Logging through MLflow (rather than a direct S3 upload) keeps the artifact store as the
# single source of truth: the script is visible in the MLflow UI and the upload goes
# through the MLflow app, so no direct s3:PutObject on the artifact bucket is needed.
# log_model_artifacts preserves the local layout, so code/inference.py stays under code/.
with tempfile.TemporaryDirectory() as tmp_dir:
    os.makedirs(os.path.join(tmp_dir, "code"))
    with open(os.path.join(tmp_dir, "code", "inference.py"), "w") as f:
        f.write(INFERENCE_SCRIPT)
    mlflow_client.log_model_artifacts(mlflow_model_id, tmp_dir)

print(f"Logged code/inference.py under {logged_model.artifact_location}/code/")

#### Step 4: Log the inference specification on the MLflow model

In [ ]:
import sagemaker_mlflow

# Schema matches the InferenceSpecification parameter of the CreateModelPackage API.
inference_specification = {
    "Containers": [
        {
            "Image": xgboost_image,
            "ModelDataSource": {
                "S3DataSource": {
                    "S3Uri": logged_model.artifact_location + "/",
                    "S3DataType": "S3Prefix",
                    "CompressionType": "None",
                }
            },
            # Point the framework container at the inference script logged above.
            "Environment": {
                "SAGEMAKER_PROGRAM": "inference.py",
                "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
            },
        }
    ],
    "SupportedContentTypes": ["text/csv"],
    "SupportedResponseMIMETypes": ["text/csv"],
    "SupportedRealtimeInferenceInstanceTypes": [instance_type],
}

# Attach the specification to the MLflow model. It is stored as
# sagemaker_inference_specification.json next to the model in the artifact store, and is
# picked up automatically by Model Registry sync when a model is registered from MLflow.
spec_uri = sagemaker_mlflow.log_inference_specification(
    mlflow_model_id, inference_specification=inference_specification
)
print(f"Inference specification logged to {spec_uri}")

#### Step 5: Register into the shared Model Package Group

In [ ]:
response = sagemaker_client.list_model_package_groups(CrossAccountFilterOption="CrossAccount")
model_package_group_arn = response['ModelPackageGroupSummaryList'][0]['ModelPackageGroupArn']
print(model_package_group_arn)

In [ ]:
create_model_package_response = sagemaker_client.create_model_package(
    ModelPackageGroupName=model_package_group_arn,
    ModelPackageDescription="Model to detect credit risk",
    ModelApprovalStatus="PendingManualApproval",
    InferenceSpecification=inference_specification,
)
model_package_arn = create_model_package_response["ModelPackageArn"]
print(f"ModelPackage Version ARN : {model_package_arn}")

# The Model Package is born deployable: the inference specification travelled with it, so
# no post-registration update_model_package call is needed.
described = sagemaker_client.describe_model_package(ModelPackageName=model_package_arn)
container = described["InferenceSpecification"]["Containers"][0]
print(f"Image:       {container['Image']}")
print(f"Model data:  {container['ModelDataSource']['S3DataSource']['S3Uri']}")
print(f"Approval:    {described['ModelApprovalStatus']}")